In [ ]:
%pip install sparknlp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 743.3/743.3 kB 37.2 MB/s eta 0:00:00


In [ ]:
from pyspark.ml import Pipeline

In [ ]:
# Non-sparknlp approach
# from pyspark.sql import SparkSession
# spark = (SparkSession.
#   builder.
#   appName('my_nlp_project').
#   getOrCreate()
# )

In [ ]:
import sparknlp

In [ ]:
spark = sparknlp.start()

In [ ]:
type(spark)

pyspark.sql.session.SparkSession

In [ ]:
import pandas as pd

In [ ]:
pd_doc_df = pd.DataFrame(
    {'text': [
        "My family and I loved the Nathan Fielder movie! I hope to invest in Theranos ASAP",
        "I love clay and I love faces, so Clayface was just perfect! 5 bags of popcorn",
        "I hated Obsession. I hate obsessions in general.",
        "This is also a sentence!",
    ],
     'review': [5, 5, 1, 3]
  }
)
pd_doc_df

,text,review
0,I love Minecraft movie! Chkn jockeyyyyyyyyyyyy...,5
1,Ana went to see Tron Ares. She loved it!,5
2,I hated Tron Ares.,1
3,This is also a sentence!,3


In [ ]:
doc_df = spark.createDataFrame(pd_doc_df)
# doc_df = spark.createDataFrame("s3://mybigdata/huge-data-files/*")

In [ ]:
# Lazy evaluation
doc_df

DataFrame[text: string, review: bigint]

In [ ]:
# Eager evaluation
doc_df.show(truncate=True, n=5)

+--------------------+------+
|                text|review|
+--------------------+------+
|I love Minecraft ...|     5|
|Ana went to see T...|     5|
|  I hated Tron Ares.|     1|
|This is also a se...|     3|
+--------------------+------+



In [ ]:
doc_df.show(truncate=False, vertical=True, n=5)

-RECORD 0----------------------
 text   | I love Minecraft ... 
 review | 5                    
-RECORD 1----------------------
 text   | Ana went to see T... 
 review | 5                    
-RECORD 2----------------------
 text   | I hated Tron Ares.   
 review | 1                    
-RECORD 3----------------------
 text   | This is also a se... 
 review | 3                    



## Transforming into a SparkNLP Document

In [ ]:
from sparknlp.base import DocumentAssembler

In [ ]:
my_doc_assembler = (
  DocumentAssembler()
  .setInputCol("text")
  .setOutputCol("document")
)

In [ ]:
my_doc_assembler

DocumentAssembler_f04fc1846014

In [ ]:
# Lazy evaluation
result = my_doc_assembler.transform(doc_df)
result

DataFrame[text: string, review: bigint, document: array<struct<annotatorType:string,begin:int,end:int,result:string,metadata:map<string,string>,embeddings:array<float>>>]

In [ ]:
# Eager evaluation
result.show(truncate=False, vertical=True)

-RECORD 0------------------------------------------------------------------------------------------------------------------------------------------------------------
 text     | Spark NLP is an open-source text processing library.                                                                                                     
 document | [{document, 0, 51, Spark NLP is an open-source text processing library., {sentence -> 0}, []}]                                                           
-RECORD 1------------------------------------------------------------------------------------------------------------------------------------------------------------
 text     | Ana went to see Tron Ares. She love it, she want Jared Leto to win every Oscar. And you do too, dear student.                                            
 document | [{document, 0, 108, Ana went to see Tron Ares. She love it, she want Jared Leto to win every Oscar. And you do too, dear student., {sentence -> 0}, []}] 
-REC

In [ ]:
result.select("document").show(truncate=False, vertical=True)

-RECORD 0------------------------------------------------------------------------------------------------------------------------------------------------------------
 document | [{document, 0, 51, Spark NLP is an open-source text processing library., {sentence -> 0}, []}]                                                           
-RECORD 1------------------------------------------------------------------------------------------------------------------------------------------------------------
 document | [{document, 0, 108, Ana went to see Tron Ares. She love it, she want Jared Leto to win every Oscar. And you do too, dear student., {sentence -> 0}, []}] 
-RECORD 2------------------------------------------------------------------------------------------------------------------------------------------------------------
 document | [{document, 0, 17, I hated Tron Ares., {sentence -> 0}, []}]                                                                                             
-REC

## Tokenization

https://sparknlp.org/api/python/reference/autosummary/sparknlp/annotator/token/tokenizer/index.html

In [ ]:
from sparknlp.annotator import Tokenizer

In [ ]:
my_tokenizer = (
    Tokenizer()
    .setInputCols(["document"])
    .setOutputCol("token")
    .fit(doc_df)
)

In [ ]:
my_tokenizer

REGEX_TOKENIZER_d4ca102f0abd

But, now that we have **two** stages (document assembly and tokenization), we should start using the `Pipeline` object provided by `pyspark.ml`!

In [ ]:
from pyspark.ml import Pipeline

In [ ]:
my_nlp_pipeline = (
    Pipeline()
    .setStages([
        my_doc_assembler,
        my_tokenizer,
    ])
    .fit(doc_df)
)
my_nlp_pipeline

PipelineModel_f148c5de713e

In [ ]:
nlp_pipeline_result = my_nlp_pipeline.transform(doc_df)
nlp_pipeline_result

DataFrame[text: string, document: array<struct<annotatorType:string,begin:int,end:int,result:string,metadata:map<string,string>,embeddings:array<float>>>, token: array<struct<annotatorType:string,begin:int,end:int,result:string,metadata:map<string,string>,embeddings:array<float>>>]

In [ ]:
nlp_pipeline_result.show(truncate=False, vertical=True, n = 3)

-RECORD 0-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Normalization

https://sparknlp.org/api/python/reference/autosummary/sparknlp/annotator/normalizer/index.html

In [ ]:
from sparknlp.annotator import Normalizer

In [ ]:
my_normalizer = (
    Normalizer()
    .setInputCols(["token"])
    .setOutputCol("normalized")
    .setLowercase(True)
    .setMinLength(2)
)

Our new pipeline is: assembly &rarr; tokenization &rarr; normalization

In [ ]:
my_nlp_pipeline = Pipeline()
my_nlp_pipeline.setStages([
    my_doc_assembler,
    my_tokenizer,
    my_normalizer,
])
pipeline_result = my_nlp_pipeline.fit(doc_df).transform(doc_df)
pipeline_result

DataFrame[text: string, document: array<struct<annotatorType:string,begin:int,end:int,result:string,metadata:map<string,string>,embeddings:array<float>>>, token: array<struct<annotatorType:string,begin:int,end:int,result:string,metadata:map<string,string>,embeddings:array<float>>>, normalized: array<struct<annotatorType:string,begin:int,end:int,result:string,metadata:map<string,string>,embeddings:array<float>>>]

In [ ]:
# Eager evaluation
pipeline_result.show(truncate=False, vertical=True, n=4)

-RECORD 0-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## The `Finisher` Class

In [ ]:
from sparknlp.base import Finisher

In [ ]:
my_finisher = (Finisher()
    .setInputCols(["normalized"])
    #.setIncludeMetadata(True)
)

In [ ]:
my_nlp_pipeline = Pipeline()
my_nlp_pipeline.setStages([
    my_doc_assembler,
    my_tokenizer,
    my_normalizer,
    my_finisher,
])
pipeline_result = my_nlp_pipeline.fit(doc_df).transform(doc_df)
pipeline_result

DataFrame[text: string, finished_normalized: array<string>]

In [ ]:
# Eager evaluation
pipeline_result.show(truncate=False, vertical=True)

-RECORD 0-----------------------------------------------------------------------------------------------------------------------------------------------
 text                | Spark NLP is an open-source text processing library.                                                                             
 finished_normalized | [spark, nlp, is, an, opensource, text, processing, library]                                                                      
-RECORD 1-----------------------------------------------------------------------------------------------------------------------------------------------
 text                | Ana went to see Tron Ares. She love it, she want Jared Leto to win every Oscar. And you do too, dear student.                    
 finished_normalized | [ana, went, to, see, tron, ares, she, love, it, she, want, jared, leto, to, win, every, oscar, and, you, do, too, dear, student] 
-RECORD 2-------------------------------------------------------------------------

## Sentiment Analysis

https://www.johnsnowlabs.com/sentiment-analysis-using-a-naive-bayes-model/

https://sparknlp.org/api/python/reference/autosummary/sparknlp/annotator/sentiment/vivekn_sentiment/index.html

In [ ]:
from sparknlp.annotator import ViveknSentimentModel

In [ ]:
# %pip install mycustomnlpthing
# %pip install git+https://github.com/jeff/coolcustomnlpclass

In [ ]:
# Step 4: Sentiment Detection
my_sentiment_extractor = (
    ViveknSentimentModel
    .pretrained('sentiment_vivekn')
    .setInputCols(["document", "normalized"])
    .setOutputCol("sentiment")
    # .setFeatureLimit(4)
)


sentiment_vivekn download started this may take some time.
Approximate size to download 873.6 KB
[OK!]


First without the finisher, so we can see the "raw" sentiment output

In [ ]:
my_sentiment_pipeline = Pipeline(stages=[
  my_doc_assembler,
  my_tokenizer,
  my_normalizer,
  my_sentiment_extractor
])
sentiment_result = my_sentiment_pipeline.fit(doc_df).transform(doc_df)
sentiment_result.show(truncate=False, vertical=True)

-RECORD 0-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
my_sentiment_finisher= (
    Finisher()
    .setIncludeMetadata(True)
    .setInputCols(["sentiment"])
    .setOutputCols("final_sentiment")
)

my_sentiment_pipeline = Pipeline(stages=[
  my_doc_assembler,
  my_tokenizer,
  my_normalizer,
  my_sentiment_extractor,
  my_sentiment_finisher
])
sentiment_result = my_sentiment_pipeline.fit(doc_df).transform(doc_df)
sentiment_result

DataFrame[text: string, final_sentiment: array<string>, final_sentiment_metadata: array<struct<_1:string,_2:string>>]

In [ ]:
sentiment_result.show(truncate=False, vertical=True)

-RECORD 0---------------------------------------------------------------------------------------------------------------------------------
 text                     | Spark NLP is an open-source text processing library.                                                          
 final_sentiment          | [positive]                                                                                                    
 final_sentiment_metadata | [{confidence, 0.7283}]                                                                                        
-RECORD 1---------------------------------------------------------------------------------------------------------------------------------
 text                     | Ana went to see Tron Ares. She love it, she want Jared Leto to win every Oscar. And you do too, dear student. 
 final_sentiment          | [positive]                                                                                                    
 final_sentiment_metadata |